# ODS v2: Travel quotes

Builds 1 ods tables in `ods_v2`: `QuoteTravel`.

Needs these notebooks to have succeeded first: 20_Quote.

Generated by `ods_build/v2/build_notebook.py` from `entities/*.yaml`: one SQL statement per cell.

**If a cell fails**

1. Read the failing cell's header: what it builds, what it reads, its grain, key and dedupe rule.
2. Fix the cause. Most failures are a source change (a renamed or missing column) or a broken key (duplicates).
   The checks cell at the end names the table and the likely cause.
3. Re-run from the failed cell: every cell rebuilds its table completely (`create or replace`), so re-running is
   always safe. After a session restart, run the first three cells again first (parameters, setup, start log).
4. **Copy the fix back into the config** (`ods_build/v2/entities/*.yaml`) and regenerate. This notebook is generated:
   a fix made only here is lost at the next generation (the drift check lists such cells).


In [ ]:
%%pyspark
# Parameters (set by ODS_v2_Run; leave blank when running by hand)
batch_id   = ""
source_set = ""


In [ ]:
%%pyspark
%run ODS_v2_00_Setup

In [ ]:
%%pyspark
ods_run_log("23_Quote_Travel", "started")


## QuoteTravel


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.QuoteTravel                                                                  */
/*  Description : Travel quote detail, latest update per quote. Names, address, date of birth are not t */
/*              : aken; email and mobile are                                                            */
/*              : salted hashes. MarketingConsent_* are consent flags, not contact details.             */
/*              :                                                                                       */
/*              : Grain: one row per travel quote (quote number; incomplete quotes by hashed email + tr */
/*              : avel date)                                                                            */
/*              : Business key: SourceSystemQuoteReference                                              */
/*              : History: current   Cut-off: QuoteDate (through Quote)                                 */
/*              : Source: dlk.ext_travel_quotes                                                         */
/*              : Reads: dlk.ext_travel_quotes, ods_v2.Quote                                            */
/******************************************************************************************************/
create or replace table ${ods.target}.QuoteTravel as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  q.QuoteId,
                q.SourceSystemQuoteReference,
                q.PolicyId,
                t.quote_number                                                  as QuoteNumber,
                t.travel_certificatestatus                                      as CertificateStatus,
                nullif(upper(trim(t.certificate_reference)), '')                                   as CertificateReference,
                t.cover_level                                                   as CoverLevel,
                t.destination                                                   as Destination,
                t.no_adults                                                     as NoAdults,
                t.no_children                                                   as NoChildren,
                t.no_infants                                                    as NoInfants,
                t.travel_premium                                                as TravelPremium,
                t.travel_quoteDate                                              as QuoteDate,
                t.travel_startdate                                              as TravelStartDate,
                t.travel_enddate                                                as TravelEndDate,
                t.travel_purchaseDate                                           as PurchaseDate,
                t.business_type                                                 as BusinessType,
                t.MarketingConsent_Email,
                t.MarketingConsent_SMS,
                t.Marketing_Opted_In                                            as MarketingOptedIn,
                t.reseller                                                      as Reseller,
                t.campaign                                                      as Campaign,
                t.phi                                                           as Phi,
                t.ski                                                           as Ski,
                t.has_medical                                                   as HasMedical,
                t.travel_health_insurance                                       as TravelHealthInsurance,
                t.duration                                                      as Duration,
                t.customer_age                                                  as CustomerAge,
                t.LastUpdated,
                ods_hash_email(t.email)                                             as EmailKey,
                ods_hash_phone(t.mobile_number)                                     as MobileKey,
                ods_hash_id(t.user_id, '')                                              as UserIdKey
        from    (
                select  *,
                        case when quote_number <> 0 then cast(quote_number as string)
                             else concat('E:', ods_hash_email(email), ':', date_format(travel_startdate, 'yyyyMMdd')) end as Ref,
                        row_number() over (partition by case when quote_number <> 0 then cast(quote_number as string)
                                                             else concat(lower(trim(email)), date_format(travel_startdate, 'yyyyMMdd')) end
                                           order by LastUpdated desc, EXT_Travel_QuotesKey desc) as rn
                from    ${ods.dlk}.ext_travel_quotes
                ) t
        join    ${ods.target}.Quote q   on q.SourceSystemId = 8 and q.SourceSystemQuoteReference = t.Ref
        where   t.rn = 1
        ) x

## Checks


In [ ]:
%%pyspark
# Checks for the tables this notebook built. All are checked and logged to OdsBuildLog; the cell then fails if
# any check failed, naming the table and the likely cause.
import json, re
from functools import reduce
from pyspark.sql import functions as F
SPEC = json.loads(r"""[{"table": "QuoteTravel", "bk": ["SourceSystemQuoteReference"], "sk": null, "nullable": [], "allow": ["MarketingConsent_Email", "MarketingConsent_SMS"]}]""")
PII_NAME = re.compile(r"(e_?mail|forename|first_?name|surname|last_?name|full_?name|second_?name|phone|mobile|telephone|post_?code|eircode(?!_?area)|address_?line|risk_?address|postal_?address|dob$|date_?of_?birth|licen[cs]e_?num|registration_?no|reg_?number|registration_number|^ani$|^dnis$|originatingdnis|^payer_|iban|^bic|^contact$|rendered_?message)", re.I)
EMAIL_RE = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
MOBILE_RE = r"(^|[^0-9A-Za-z-])(\+?353 ?|0)8[35-9][ -]?[0-9]{3}[ -]?[0-9]{4}([^0-9A-Za-z-]|$)"   # stands alone, not inside an id
rows, failures = [], []
for s in SPEC:
    t = f"{target_schema}.{s['table']}"
    df = spark.table(t)
    bk = s["bk"]
    must = [c for c in bk if c not in s["nullable"]]
    agg = df.agg(F.count(F.lit(1)).alias("n"),
                 F.countDistinct(F.struct(*bk)).alias("d_bk"),
                 F.sum(F.when(reduce(lambda a, b: a | b, [F.col(c).isNull() for c in must] or [F.lit(False)]), 1).otherwise(0)).alias("null_bk"),
                 *( [F.countDistinct(s["sk"]).alias("d_sk")] if s["sk"] else [] )).first()
    pii_cols = [c for c in df.columns if PII_NAME.search(c) and not c.endswith("Key") and c not in s["allow"]]
    text_cols = [f.name for f in df.schema.fields if f.dataType.simpleString() == "string"]   # hashes never match
    hits = {}
    if text_cols:
        found = df.select(*text_cols).limit(1000000).agg(
            *[F.sum((F.col(c).rlike(EMAIL_RE) | F.col(c).rlike(MOBILE_RE)).cast("int")).alias(c) for c in text_cols]).first().asDict()
        hits = {c: v for c, v in found.items() if v}
    problems, advice = [], []
    keys = " + ".join(bk)
    if agg["n"] == 0:
        problems.append("empty")
        advice.append("the table is empty: its source is empty, or a filter (cut-off date, join) removed every row")
    if agg["d_bk"] != agg["n"]:
        problems.append(f"{agg['n'] - agg['d_bk']:,} duplicate business keys")
        advice.append(f"several rows share a key ({keys}): the source sent the same record more than once, or the "
                      f"dedupe rule in the cell no longer holds. Examples: select {', '.join(bk)}, count(*) from {t} "
                      f"group by {', '.join(bk)} having count(*) > 1")
    if agg["null_bk"]:
        problems.append(f"{agg['null_bk']:,} null business keys")
        advice.append(f"key columns ({', '.join(must)}) are empty on some rows: if that is legitimate for a column, add it "
                      f"to key_nullable for this table in the config; otherwise the source changed")
    if s["sk"] and agg["d_sk"] != agg["n"]:
        problems.append("surrogate key not unique")
        advice.append("the surrogate key repeats: the business key is not unique (see above)")
    if pii_cols:
        problems.append(f"PII-named columns {pii_cols}")
        advice.append("columns named like personal data: hash them (HASH_*) or leave them out in the config")
    if hits:
        problems.append(f"email/mobile values in {hits}")
        advice.append("emails / mobile numbers inside these columns: wrap the column in REDACT() (or hash it) in the config")
    rows.append((batch_id, s["table"], agg["n"], agg["n"] - agg["d_bk"], agg["null_bk"], "; ".join(problems) or "ok"))
    print(f"{s['table']:<28} {agg['n']:>13,}  {'; '.join(problems) or 'ok'}")
    if problems:
        failures.append(f"{s['table']}: " + " | ".join(advice))
log = spark.createDataFrame(rows, "OdsBatchId string, TableName string, RowCount long, DuplicateKeys long, NullKeys long, Result string")
log.withColumn("CheckedAt", F.current_timestamp()).write.mode("append").saveAsTable(f"{target_schema}.OdsBuildLog")
if failures:
    ods_run_log("23_Quote_Travel", "failed", "; ".join(f.split(":")[0] for f in failures))
    raise AssertionError("ods checks failed:\n- " + "\n- ".join(failures))
ods_run_log("23_Quote_Travel", "succeeded")
